# GNN Illustration

Train one depth-2 GIN model and visualize the requested exemplar organoid as a 3D cell graph. The first graph colors cells by marker identity using the same marker palette as the mesh figures; the second colors cells by the first principal component of the final local node embedding returned by the GNN.

In [18]:
import copy
import json
import pickle
import random
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_ROOT = PROJECT_ROOT / "training_data"
sys.path.append(str(PROJECT_ROOT))

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "legend.frameon": False,
})

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_ROOT    =", DATA_ROOT)
print("torch        =", torch.__version__)
print("cuda         =", torch.cuda.is_available())


PROJECT_ROOT = /home/fmoller/Projects/LearningOrganoids/GraphNN
DATA_ROOT    = /home/fmoller/Projects/LearningOrganoids/GraphNN/training_data
torch        = 2.8.0+cu128
cuda         = True


**Settings**

In [19]:
EXPERIMENT_GROUP = "GNN_illustration"
DATASET_NAME = "after_cleanup"
TARGET_INDICES = [0]
TARGET_INDEX_FOR_ANALYSIS = 0

EXEMPLAR_KEY = ("20250929", "day4p5_B06_49")
FORCED_VAL_KEYS = {EXEMPLAR_KEY}
PRESERVE_FORCED_EXEMPLAR = True

USE_GLOBAL_FEATURES = True
FILTER_BLACKLISTED_ORGANOIDS = True
TIMEPOINT_FILTER_MODE = "rest"  # "rest", "day3p5", or "all"
DAY3P5_TIMEPOINT = "day3p5"
FILL_MISSING_COMPLEXITY = True
MISSING_COMPLEXITY_GROUP = {
    "dataset": "20251201",
    "timepoint": "day4p5",
    "fill_value": 2.1,
}
SPHERICITY_MAX = 0.92
COMPLEXITY_MIN = 2.0
ALLOW_MISSING_COMPLEXITY = False
INTERPOLATE_TARGET_OUTLIERS = True
OUTLIER_CLIP_QUANTILES = (0.005, 0.995)

VAL_FRAC = 0.2
SPLIT_SEED = 42
MODEL_DEPTH = 2
HIDDEN_DIM = 4 * 64
DROPOUT = 0.1
NORM = "batch"
RESIDUAL = True
LR = 3e-4
BATCH_SIZE = 128
MAX_EPOCHS = 500
PATIENCE = 30
NUM_WORKERS = 4
EDGE_LOSS_WEIGHT = 0.20
EDGE_LOSS_PARAMS = {
    "weighted": False,
    "alpha": 2.0,
    "normalize_by": "graph_std",
    "clip_weight": 4.0,
}

GRAPH_NODE_SIZE = 5
GRAPH_EDGE_WIDTH = 2.5
GRAPH_WIDTH = 780
GRAPH_HEIGHT = 600


GRAPH_CAMERA_BASE = dict(eye=dict(x=-0.8, y=-0.8, z=1.2))
GRAPH_CAMERA_ZOOM = 1.25          # >1 zooms out
GRAPH_CAMERA_ROTATE_DEG = 30      # use -30 if this goes opposite of mouse-drag-right

def rotate_camera_about_z(camera, degrees, zoom=1.0):
    eye = camera["eye"]
    theta = np.deg2rad(degrees)

    x, y, z = float(eye["x"]), float(eye["y"]), float(eye["z"])
    x_rot = x * np.cos(theta) - y * np.sin(theta)
    y_rot = x * np.sin(theta) + y * np.cos(theta)

    return dict(eye=dict(
        x=float(zoom * x_rot),
        y=float(zoom * y_rot),
        z=float(zoom * z),
    ))

GRAPH_CAMERA = rotate_camera_about_z(
    GRAPH_CAMERA_BASE,
    GRAPH_CAMERA_ROTATE_DEG,
    zoom=GRAPH_CAMERA_ZOOM,
)

GRAPH_NORMALIZE_COORDINATES = True
MARKER_THRESHOLD = 0.5
ENCODING_COLOR_MODE = "pca1"

RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_NAME = f"depth{MODEL_DEPTH}_{EXEMPLAR_KEY[0]}_{EXEMPLAR_KEY[1]}_{RUN_TIMESTAMP}"
SAVE_DIR = PROJECT_ROOT / "results_experiments" / EXPERIMENT_GROUP / RUN_NAME
FIGURES_DIR = SAVE_DIR / "figures"
TABLES_DIR = SAVE_DIR / "tables"
ARRAYS_DIR = SAVE_DIR / "arrays"
for directory in (FIGURES_DIR, TABLES_DIR, ARRAYS_DIR):
    directory.mkdir(parents=True, exist_ok=True)


def set_all_seeds(seed):
    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))


set_all_seeds(SPLIT_SEED)


**Helpers**

In [20]:
def safe_filename(name):
    text = str(name).strip().replace("/", "_")
    cleaned = "".join(
        char if (char.isalnum() or char in "._-") else "_" for char in text
    ).strip("._-")
    while "__" in cleaned:
        cleaned = cleaned.replace("__", "_")
    return cleaned or "figure"


def save_plotly_figure(fig, name, *, scale=2):
    stem = safe_filename(name)
    html_path = FIGURES_DIR / f"{stem}.html"
    fig.write_html(str(html_path), include_plotlyjs="cdn")
    for suffix in ("png", "svg"):
        try:
            fig.write_image(str(FIGURES_DIR / f"{stem}.{suffix}"), scale=scale)
        except Exception as exc:
            print(f"Plotly {suffix.upper()} export skipped for {stem}: {exc}")
    return html_path


def jsonable(obj):
    if isinstance(obj, dict):
        return {str(key): jsonable(value) for key, value in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [jsonable(value) for value in obj]
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, np.generic):
        return obj.item()
    if isinstance(obj, torch.Tensor):
        return obj.detach().cpu().tolist()
    if isinstance(obj, (torch.dtype, np.dtype)):
        return str(obj)
    return obj


def infer_target_dim(graphs_in):
    y = graphs_in[0].y
    return 1 if y.ndim == 1 else int(y.shape[1])


def select_target_column(values, target_index=0):
    array = np.asarray(values)
    if array.ndim == 1:
        return array.reshape(-1)
    if array.ndim == 2 and array.shape[1] == 1:
        return array[:, 0]
    if array.ndim == 2:
        return array[:, int(target_index)]
    raise ValueError(f"Expected 1-D or 2-D values, got {array.shape}.")


**Load And Filter Data**

In [21]:
from src.data.filters import (
    filter_graphs_by_blacklist,
    filter_graphs_by_metadata,
    filter_graphs_by_numeric_metadata,
    filter_graphs_by_sphericity,
    load_graph_blacklist_from_dir,
)
from src.data.io import load_graph_dataset_from_dir, select_graph_targets
from src.data.metadata import (
    attach_metadata_to_graphs,
    fill_missing_metadata_for_group,
    load_aux_metadata_for_dir,
    load_marker_names_from_dir,
)
from src.data.preprocessing import interpolate_target_outliers_from_neighbors
from src.data.splits import graph_metadata_key, split_graphs_by_keys


def dedupe_graphs_by_metadata_key(graphs_in):
    seen = set()
    out = []
    for graph in graphs_in:
        key = graph_metadata_key(graph)
        if key in seen:
            continue
        seen.add(key)
        out.append(graph)
    return out


data_dir = DATA_ROOT / DATASET_NAME
graphs = load_graph_dataset_from_dir(str(data_dir))
metadata = load_aux_metadata_for_dir(str(data_dir))
attach_metadata_to_graphs(graphs, metadata, exclude_keys=None)
graphs = select_graph_targets(graphs, target_indices=TARGET_INDICES, inplace=False)
marker_names = load_marker_names_from_dir(str(data_dir))
if marker_names is None:
    marker_names = [f"marker_{index}" for index in range(int(graphs[0].x.size(1)))]
marker_names = list(marker_names)
print(f"Loaded {len(graphs)} organoids and {len(marker_names)} markers.")

if PRESERVE_FORCED_EXEMPLAR:
    forced_graphs, graphs_to_filter = split_graphs_by_keys(
        graphs,
        FORCED_VAL_KEYS,
        key_fn=graph_metadata_key,
        inplace=False,
    )
    if len(forced_graphs) == 0:
        raise ValueError(f"Forced exemplar {EXEMPLAR_KEY} was not found before filtering.")
    print(f"Preserving {len(forced_graphs)} forced exemplar graph(s) through filtering.")
else:
    forced_graphs, graphs_to_filter = [], graphs

if TIMEPOINT_FILTER_MODE == "rest":
    graphs_to_filter = filter_graphs_by_metadata(
        graphs_to_filter,
        key="timepoint",
        drop_values={DAY3P5_TIMEPOINT},
        missing="keep",
        inplace=False,
        print_summary=True,
    )
elif TIMEPOINT_FILTER_MODE == "day3p5":
    graphs_to_filter = filter_graphs_by_metadata(
        graphs_to_filter,
        key="timepoint",
        keep_values={DAY3P5_TIMEPOINT},
        missing="drop",
        inplace=False,
        print_summary=True,
    )
elif TIMEPOINT_FILTER_MODE not in (None, "all"):
    raise ValueError('TIMEPOINT_FILTER_MODE must be "rest", "day3p5", or "all".')

if FILTER_BLACKLISTED_ORGANOIDS:
    blacklist = load_graph_blacklist_from_dir(data_dir)
    graphs_to_filter = filter_graphs_by_blacklist(
        graphs_to_filter,
        blacklist,
        print_summary=True,
    )
else:
    blacklist = set()

if FILL_MISSING_COMPLEXITY:
    graphs_to_filter = fill_missing_metadata_for_group(
        graphs_to_filter,
        field="complexity",
        fill_value=MISSING_COMPLEXITY_GROUP["fill_value"],
        dataset=MISSING_COMPLEXITY_GROUP["dataset"],
        timepoint=MISSING_COMPLEXITY_GROUP["timepoint"],
    )
    forced_graphs = fill_missing_metadata_for_group(
        forced_graphs,
        field="complexity",
        fill_value=MISSING_COMPLEXITY_GROUP["fill_value"],
        dataset=MISSING_COMPLEXITY_GROUP["dataset"],
        timepoint=MISSING_COMPLEXITY_GROUP["timepoint"],
    )

if SPHERICITY_MAX is not None:
    graphs_to_filter = filter_graphs_by_sphericity(
        graphs_to_filter,
        max_sphericity=SPHERICITY_MAX,
        print_summary=True,
    )

if COMPLEXITY_MIN is not None:
    graphs_to_filter = filter_graphs_by_numeric_metadata(
        graphs_to_filter,
        key="complexity",
        min_value=COMPLEXITY_MIN,
        allow_missing=ALLOW_MISSING_COMPLEXITY,
        print_summary=True,
    )

graphs = dedupe_graphs_by_metadata_key(graphs_to_filter + forced_graphs)
print(f"Graphs after filtering and forced-exemplar restoration: {len(graphs)}")

if INTERPOLATE_TARGET_OUTLIERS:
    graphs, interpolation_info = interpolate_target_outliers_from_neighbors(
        graphs,
        target_indices=None,
        clip_quantiles=OUTLIER_CLIP_QUANTILES,
        inplace=False,
        report=True,
    )
else:
    interpolation_info = None


Loaded 1500 graphs; skipped 0.
Loaded 1500 organoids and 7 markers.
Preserving 1 forced exemplar graph(s) through filtering.
filter_graphs_by_metadata(key='timepoint'): kept 1209 / 1499 graphs

dataset          timepoint          kept  total     frac
----------------------------------------------------------
20250929         day3p5                0    290    0.000
20250929         day4                342    342    1.000
20250929         day4p5              112    112    1.000
20250929         day4p5-more         477    477    1.000
20251201         day4p5              278    278    1.000
filter_graphs_by_blacklist(n_keys=17): kept 1192 / 1209 graphs

dataset          timepoint          kept  total     frac
----------------------------------------------------------
20250929         day4                330    342    0.965
20250929         day4p5              111    112    0.991
20250929         day4p5-more         475    477    0.996
20251201         day4p5              276    278    0.9

**Metadata Features And Split**

In [22]:
from src.data.metadata import (
    add_log_metadata_features,
    infer_global_dim,
    promote_metadata_to_graph_tensors,
    snapshot_graph_metadata,
    strip_graph_metadata,
)
from src.data.splits import train_val_split_graphs
from src.data.target_transforms import (
    AsinhStandardizeTransform,
    standardize_graph_global_features,
)

FIELD_SPECS = [{
    "meta_keys": [
        "log_surface_area",
        "log_volume",
        "log_volume_over_area",
        "log_num_cells",
    ],
    "attr_name": "global_feat",
    "kind": "graph_vector",
    "dtype": torch.float32,
}]

if USE_GLOBAL_FEATURES:
    graphs = promote_metadata_to_graph_tensors(
        add_log_metadata_features(graphs, inplace=False),
        FIELD_SPECS,
        inplace=False,
    )
else:
    raise ValueError("This notebook currently expects USE_GLOBAL_FEATURES=True.")

g_train_raw, g_val_raw, split_info = train_val_split_graphs(
    graphs,
    val_frac=VAL_FRAC,
    seed=SPLIT_SEED,
    force_val_keys=FORCED_VAL_KEYS,
    key_fn=graph_metadata_key,
    inplace=False,
)
print(f"Split -> train: {len(g_train_raw)} | val: {len(g_val_raw)}")
print("Forced validation keys:", split_info.get("forced_val_keys"))

train_meta_lookup = snapshot_graph_metadata(g_train_raw)
val_meta_lookup = snapshot_graph_metadata(g_val_raw)
g_train = strip_graph_metadata(g_train_raw, inplace=False)
g_val = strip_graph_metadata(g_val_raw, inplace=False)

standardize_graph_global_features(
    g_train,
    g_val,
    attr_name="global_feat",
    robust=False,
)
target_transform = AsinhStandardizeTransform(robust=True)
target_transform.fit(g_train)
target_transform.transform_graphs(g_train, in_place=True)
target_transform.transform_graphs(g_val, in_place=True)

split_summary_df = pd.DataFrame([{
    "n_graphs_filtered": len(graphs),
    "n_train_graphs": len(g_train),
    "n_val_graphs": len(g_val),
    "n_train_nodes": sum(int(graph.x.shape[0]) for graph in g_train),
    "n_val_nodes": sum(int(graph.x.shape[0]) for graph in g_val),
    "val_frac": VAL_FRAC,
    "split_seed": SPLIT_SEED,
    "forced_val_keys": repr(sorted(FORCED_VAL_KEYS)),
}])
split_summary_df.to_csv(TABLES_DIR / "split_summary.csv", index=False)
display(split_summary_df)


Split -> train: 149 | val: 37
Forced validation keys: [('20250929', 'day4p5_B06_49')]


,n_graphs_filtered,n_train_graphs,n_val_graphs,n_train_nodes,n_val_nodes,val_frac,split_seed,forced_val_keys
0,186,149,37,77519,23162,0.2,42,"[('20250929', 'day4p5_B06_49')]"


**Train Depth-2 GNN**

In [23]:
from src.models.gnn import GINCurvature
from src.training.loop import TrainConfig, train
from src.training.losses import WeightedLossTerm, edge_loss_term


def make_gin_model(graphs_in, depth):
    return GINCurvature(
        n_markers=int(graphs_in[0].x.size(1)),
        global_dim=infer_global_dim(graphs_in),
        hidden_dim=HIDDEN_DIM,
        num_layers=int(depth),
        dropout=DROPOUT,
        residual=RESIDUAL,
        norm=NORM,
        target_dim=infer_target_dim(graphs_in),
    )

cfg = TrainConfig(
    lr=LR,
    batch_size=BATCH_SIZE,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    num_workers=NUM_WORKERS,
    aux_losses=[WeightedLossTerm(
        name="edge",
        fn=edge_loss_term,
        weight=EDGE_LOSS_WEIGHT,
        params=EDGE_LOSS_PARAMS,
    )],
)
device = cfg.device
set_all_seeds(SPLIT_SEED + MODEL_DEPTH)
model = make_gin_model(g_train, MODEL_DEPTH)
model, metrics, history = train(model, g_train, g_val, cfg)
training_summary_df = pd.DataFrame([{
    "model": "GINCurvature",
    "depth": MODEL_DEPTH,
    "seed": SPLIT_SEED + MODEL_DEPTH,
    "val_mae_transformed": float(metrics["val_mae"]),
    "epochs_trained": len(history["val_mae"]),
}])
training_summary_df.to_csv(TABLES_DIR / "training_summary.csv", index=False)
display(training_summary_df)


epoch 001 | train loss 0.5910 mae 0.8030 | val loss 0.5552 mae 0.8022
epoch 002 | train loss 0.5106 mae 0.7620 | val loss 0.5540 mae 0.7974
epoch 003 | train loss 0.4770 mae 0.7343 | val loss 0.5488 mae 0.7951
epoch 004 | train loss 0.4690 mae 0.7297 | val loss 0.5291 mae 0.7790
epoch 005 | train loss 0.4497 mae 0.7165 | val loss 0.5128 mae 0.7623
epoch 006 | train loss 0.4413 mae 0.7088 | val loss 0.4959 mae 0.7496
epoch 007 | train loss 0.4395 mae 0.7092 | val loss 0.4776 mae 0.7416
epoch 008 | train loss 0.4260 mae 0.7040 | val loss 0.4712 mae 0.7397
epoch 009 | train loss 0.4189 mae 0.6977 | val loss 0.4714 mae 0.7363
epoch 010 | train loss 0.4178 mae 0.6934 | val loss 0.4641 mae 0.7309
epoch 011 | train loss 0.4088 mae 0.6896 | val loss 0.4482 mae 0.7265
epoch 012 | train loss 0.4063 mae 0.6903 | val loss 0.4428 mae 0.7272
epoch 013 | train loss 0.4065 mae 0.6921 | val loss 0.4312 mae 0.7162
epoch 014 | train loss 0.3948 mae 0.6806 | val loss 0.4312 mae 0.7108
epoch 015 | train lo

,model,depth,seed,val_mae_transformed,epochs_trained
0,GINCurvature,2,44,0.644138,291


**Select Exemplar And Extract Embeddings**

In [24]:
from src.analysis.motif_clustering import extract_node_embeddings
from src.data.metadata import get_graph_metadata
from src.data.splits import select_graphs_by_keys

exemplar_graphs = select_graphs_by_keys(
    g_val,
    FORCED_VAL_KEYS,
    key_fn=graph_metadata_key,
    meta_lookup=val_meta_lookup,
    inplace=False,
)
if len(exemplar_graphs) != 1:
    raise ValueError(f"Expected exactly one exemplar graph in validation, found {len(exemplar_graphs)}.")
exemplar_graph = exemplar_graphs[0]
exemplar_key = graph_metadata_key(exemplar_graph, meta_lookup=val_meta_lookup)
exemplar_meta = get_graph_metadata(exemplar_graph, meta_lookup=val_meta_lookup, strict=True)
print("Exemplar:", exemplar_key)
print("organoid_str:", getattr(exemplar_graph, "organoid_str", None))
print("graph_path:", exemplar_meta.get("graph_path"))

extraction = extract_node_embeddings(
    [exemplar_graph],
    model,
    device=device,
    batch_size=1,
    num_workers=0,
    strip_global_from_embedding=True,
)
embeddings_local = np.asarray(extraction.embeddings_local, dtype=np.float32)
embeddings_full = np.asarray(extraction.embeddings_full, dtype=np.float32)

scaled_embeddings = StandardScaler().fit_transform(embeddings_local)
embedding_pca = PCA(n_components=1, random_state=SPLIT_SEED)
encoding_score = embedding_pca.fit_transform(scaled_embeddings).reshape(-1)
if np.nanstd(encoding_score) > 0:
    encoding_score = (encoding_score - np.nanmean(encoding_score)) / np.nanstd(encoding_score)

np.savez_compressed(
    ARRAYS_DIR / "exemplar_node_embeddings.npz",
    embeddings_local=embeddings_local,
    embeddings_full=embeddings_full,
    encoding_score=encoding_score.astype(np.float32),
    pca_explained_variance_ratio=embedding_pca.explained_variance_ratio_.astype(np.float32),
)
print("Local embedding shape:", embeddings_local.shape)
print("PCA1 explained variance:", float(embedding_pca.explained_variance_ratio_[0]))


Exemplar: ('20250929', 'day4p5_B06_49')
organoid_str: organoid_day4p5_B06_49
graph_path: /home/fmoller/Projects/LearningOrganoids/OrganoGraph/../NicoleData/20250929/graphs_preprocessed/day4p5/day4p5_B06_49.gpickle
Local embedding shape: (710, 256)
PCA1 explained variance: 0.25989624857902527


**Graph Coordinates**

In [25]:
from organograph.mesh.OrganoidMesh import OrganoidMesh

GRAPH_COORD_ATTR_CANDIDATES = (
    "centroid",
    "centroids",
    "position",
    "pos",
    "coords",
    "coordinates",
    "coordinate",
    "center",
    "cell_center",
    "cell_centroid",
)


def _coerce_coordinate(value):
    try:
        arr = np.asarray(value, dtype=float).reshape(-1)
    except Exception:
        return None
    if arr.size < 3 or not np.all(np.isfinite(arr[:3])):
        return None
    return arr[:3]


def _load_networkx_graph_from_gpickle(path):
    path = Path(path).expanduser()
    if not path.exists():
        raise FileNotFoundError(path)
    if hasattr(nx, "read_gpickle"):
        return nx.read_gpickle(path)
    with open(path, "rb") as handle:
        return pickle.load(handle)


def coordinates_from_graph_path(metadata, n_nodes):
    graph_path = metadata.get("graph_path")
    if graph_path is None:
        raise ValueError("metadata has no graph_path")
    graph = _load_networkx_graph_from_gpickle(graph_path)
    if not hasattr(graph, "nodes"):
        raise TypeError(f"graph_path did not load a NetworkX-like graph: {type(graph)}")

    if all(node in graph.nodes for node in range(n_nodes)):
        nodes = list(range(n_nodes))
    else:
        nodes = sorted(graph.nodes)[:n_nodes]
    if len(nodes) != n_nodes:
        raise ValueError(f"graph_path has {len(nodes)} usable nodes, expected {n_nodes}")

    for attr in GRAPH_COORD_ATTR_CANDIDATES:
        coords = []
        for node in nodes:
            value = graph.nodes[node].get(attr, None)
            coord = _coerce_coordinate(value)
            if coord is None:
                coords = []
                break
            coords.append(coord)
        if len(coords) == n_nodes:
            return np.vstack(coords).astype(float), f"graph_path:{attr}"

    example_node = nodes[0]
    raise ValueError(
        "No usable coordinate attribute found in graph_path. "
        f"Example node attributes: {list(graph.nodes[example_node].keys())}"
    )


def coordinates_from_mesh_projection(metadata, n_nodes):
    mesh_path = metadata.get("mesh_path")
    proj_vertex_ids = metadata.get("proj_vertex_ids")
    if mesh_path is None or proj_vertex_ids is None:
        raise ValueError("metadata needs mesh_path and proj_vertex_ids for fallback coordinates")
    proj_vertex_ids = np.asarray(proj_vertex_ids, dtype=np.int64).reshape(-1)
    if proj_vertex_ids.shape[0] != n_nodes:
        raise ValueError(f"proj_vertex_ids length {proj_vertex_ids.shape[0]} != n_nodes {n_nodes}")
    mesh = OrganoidMesh(str(mesh_path))
    mesh.normalize_inplace()
    return np.asarray(mesh.v[proj_vertex_ids, :3], dtype=float), "mesh_path:proj_vertex_ids"


def normalize_coordinates(coords):
    coords = np.asarray(coords, dtype=float)
    coords = coords - np.nanmean(coords, axis=0, keepdims=True)
    scale = float(np.nanmax(np.linalg.norm(coords, axis=1)))
    if np.isfinite(scale) and scale > 0:
        coords = coords / scale
    return coords

n_nodes = int(exemplar_graph.x.shape[0])
try:
    coordinates, coordinate_source = coordinates_from_graph_path(exemplar_meta, n_nodes)
except Exception as exc:
    print(f"Could not use graph_path coordinates ({type(exc).__name__}: {exc}); falling back to mesh projection.")
    coordinates, coordinate_source = coordinates_from_mesh_projection(exemplar_meta, n_nodes)

if GRAPH_NORMALIZE_COORDINATES:
    coordinates = normalize_coordinates(coordinates)

print("coordinate_source:", coordinate_source)
print("coordinates:", coordinates.shape)
np.savez_compressed(
    ARRAYS_DIR / "exemplar_graph_coordinates.npz",
    coordinates=coordinates.astype(np.float32),
    coordinate_source=coordinate_source,
    graph_path=str(exemplar_meta.get("graph_path")),
    mesh_path=str(exemplar_meta.get("mesh_path")),
)


coordinate_source: graph_path:centroid
coordinates: (710, 3)


**Plotting Helpers**

In [26]:
from src.plotting.mesh_plots import marker_categories_for_graph
from src.plotting.motif_plots import DEFAULT_MARKER_COLORS


def graph_edges_xyz(edge_index, coords):
    edge_index = edge_index.detach().cpu().numpy()
    xs, ys, zs = [], [], []
    seen = set()
    for u, v in edge_index.T:
        u = int(u)
        v = int(v)
        if u == v:
            continue
        key = tuple(sorted((u, v)))
        if key in seen:
            continue
        seen.add(key)
        xs += [coords[u, 0], coords[v, 0], None]
        ys += [coords[u, 1], coords[v, 1], None]
        zs += [coords[u, 2], coords[v, 2], None]
    return xs, ys, zs


def base_graph_figure(coords, edge_index, title):
    edge_x, edge_y, edge_z = graph_edges_xyz(edge_index, coords)
    fig = go.Figure()
    fig.add_trace(go.Scatter3d(
        x=edge_x,
        y=edge_y,
        z=edge_z,
        mode="lines",
        line=dict(color="rgba(40,40,40,0.28)", width=GRAPH_EDGE_WIDTH),
        hoverinfo="skip",
        showlegend=False,
        name="cell-cell edges",
    ))
    fig.update_layout(
        title=title,
        width=GRAPH_WIDTH,
        height=GRAPH_HEIGHT,
        margin=dict(l=10, r=10, t=60, b=10),
        scene_camera=GRAPH_CAMERA,
        scene=dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            bgcolor="rgba(0,0,0,0)",
            aspectmode="data",
        ),
    )
    return fig


def add_marker_legend(fig, marker_categories, marker_colors):
    ordered = [
        marker for marker in marker_colors
        if marker in set(marker_categories) and marker != "none"
    ]
    if "none" in set(marker_categories):
        ordered.append("none")
    for marker in ordered:
        fig.add_trace(go.Scatter3d(
            x=[None], y=[None], z=[None],
            mode="markers",
            marker=dict(size=8, color=marker_colors.get(marker, marker_colors["none"])),
            name=marker,
            showlegend=True,
        ))


def plot_marker_graph(coords, graph, marker_names):
    priority_names = [marker for marker in DEFAULT_MARKER_COLORS if marker != "none"]
    marker_categories = marker_categories_for_graph(
        graph,
        marker_names,
        threshold=MARKER_THRESHOLD,
        priority_names=priority_names,
    )
    colors = [DEFAULT_MARKER_COLORS.get(marker, DEFAULT_MARKER_COLORS["none"]) for marker in marker_categories]
    fig = base_graph_figure(
        coords,
        graph.edge_index,
        f"{EXEMPLAR_KEY[0]} | {EXEMPLAR_KEY[1]} | fate marker graph",
    )
    fig.add_trace(go.Scatter3d(
        x=coords[:, 0],
        y=coords[:, 1],
        z=coords[:, 2],
        mode="markers",
        marker=dict(size=GRAPH_NODE_SIZE, color=colors, opacity=0.96),
        customdata=np.column_stack([np.arange(coords.shape[0]), marker_categories]),
        hovertemplate="cell %{customdata[0]}<br>marker %{customdata[1]}<extra></extra>",
        showlegend=False,
        name="cells",
    ))
    add_marker_legend(fig, marker_categories, DEFAULT_MARKER_COLORS)
    return fig, marker_categories, colors


def plot_encoding_graph(coords, graph, encoding_values):
    values = np.asarray(encoding_values, dtype=float).reshape(-1)
    limit = float(np.nanmax(np.abs(values)))
    if not np.isfinite(limit) or limit == 0:
        limit = 1.0
    fig = base_graph_figure(
        coords,
        graph.edge_index,
        f"{EXEMPLAR_KEY[0]} | {EXEMPLAR_KEY[1]} | final GNN encoding ({ENCODING_COLOR_MODE})",
    )
    fig.add_trace(go.Scatter3d(
        x=coords[:, 0],
        y=coords[:, 1],
        z=coords[:, 2],
        mode="markers",
        marker=dict(
            size=GRAPH_NODE_SIZE,
            color=values,
            colorscale="viridis", # "RdBu_r"
            cmin=-limit,
            cmax=limit,
            colorbar=dict(title="embedding<br>PC1 z"),
            opacity=0.96,
        ),
        customdata=np.column_stack([np.arange(coords.shape[0]), values]),
        hovertemplate="cell %{customdata[0]}<br>embedding PC1 z=%{customdata[1]:.3f}<extra></extra>",
        showlegend=False,
        name="cells",
    ))
    return fig


**Marker Graph**

In [27]:
marker_graph_figure, marker_categories, marker_colors = plot_marker_graph(
    coordinates,
    exemplar_graph,
    marker_names,
)
save_plotly_figure(marker_graph_figure, "exemplar_graph_by_fate_markers")
marker_graph_figure.show()

marker_graph_df = pd.DataFrame({
    "vertex_or_cell_index": np.arange(len(marker_categories)),
    "marker_category": marker_categories,
    "marker_color": marker_colors,
})
marker_graph_df.to_csv(TABLES_DIR / "exemplar_graph_marker_categories.csv", index=False)


/tmp/ipykernel_656282/2441658616.py:17: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(str(FIGURES_DIR / f"{stem}.{suffix}"), scale=scale)


**Final GNN Encoding Graph**

In [28]:
encoding_graph_figure = plot_encoding_graph(
    coordinates,
    exemplar_graph,
    -encoding_score,
)
save_plotly_figure(encoding_graph_figure, "exemplar_graph_by_final_gnn_encoding")
encoding_graph_figure.show()

encoding_df = pd.DataFrame({
    "cell_index": np.arange(len(encoding_score)),
    "encoding_pca1_z": encoding_score,
})
encoding_df.to_csv(TABLES_DIR / "exemplar_graph_encoding_scores.csv", index=False)


/tmp/ipykernel_656282/2441658616.py:17: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(str(FIGURES_DIR / f"{stem}.{suffix}"), scale=scale)


**Final Curvature Prediction Mesh**

In [29]:
from organograph.plotting.meshes import plot_organoid_mesh

from src.inference.predict import predict_targets
from src.plotting.mesh_plots import project_node_quantities_to_mesh

mesh_y_true_all, mesh_y_pred_all, _ = predict_targets(
    [exemplar_graph],
    model,
    device=device,
    batch_size=1,
    num_workers=0,
    target_transform=target_transform,
)
mesh_y_true = select_target_column(mesh_y_true_all, TARGET_INDEX_FOR_ANALYSIS)
mesh_y_pred = select_target_column(mesh_y_pred_all, TARGET_INDEX_FOR_ANALYSIS)

mesh_prediction_projection = project_node_quantities_to_mesh(
    exemplar_graph,
    node_true=mesh_y_true,
    node_pred=mesh_y_pred,
    meta_lookup=val_meta_lookup,
)
mesh_prediction_values = np.asarray(mesh_prediction_projection["mesh_pred"], dtype=float)
finite_mesh_prediction = mesh_prediction_values[np.isfinite(mesh_prediction_values)]
mesh_prediction_limit = float(np.nanmax(np.abs(finite_mesh_prediction))) if finite_mesh_prediction.size else 1.0
if not np.isfinite(mesh_prediction_limit) or mesh_prediction_limit <= 0:
    mesh_prediction_limit = 1.0

mesh_prediction_figure = plot_organoid_mesh(
    mesh_prediction_projection["mesh"],
    vertex_values=mesh_prediction_values,
    backend="plotly",
    colorscale="RdBu_r",
    center_at_zero=True,
    vmin=-mesh_prediction_limit,
    vmax=mesh_prediction_limit,
    show_colorbar=True,
    fig_size=(GRAPH_WIDTH, GRAPH_HEIGHT),
)
mesh_prediction_figure.update_layout(
    title=f"{EXEMPLAR_KEY[0]} | {EXEMPLAR_KEY[1]} | predicted curvature mesh",
    width=GRAPH_WIDTH,
    height=GRAPH_HEIGHT,
    margin=dict(l=10, r=10, t=60, b=10),
    scene_camera=GRAPH_CAMERA,
    scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        bgcolor="rgba(0,0,0,0)",
        aspectmode="data",
    ),
)
save_plotly_figure(mesh_prediction_figure, "exemplar_mesh_by_predicted_curvature")
mesh_prediction_figure.show()

mesh_prediction_df = pd.DataFrame({
    "vertex_index": np.arange(len(mesh_prediction_values), dtype=np.int64),
    "cell_index": np.asarray(mesh_prediction_projection["vertex_owner"], dtype=np.int64),
    "true_curvature": np.asarray(mesh_prediction_projection["mesh_true"], dtype=float),
    "predicted_curvature": mesh_prediction_values,
})
mesh_prediction_df.to_csv(TABLES_DIR / "exemplar_mesh_prediction_vertices.csv", index=False)
np.savez_compressed(
    ARRAYS_DIR / "exemplar_mesh_prediction_fields.npz",
    true_curvature=np.asarray(mesh_prediction_projection["mesh_true"], dtype=np.float32),
    predicted_curvature=mesh_prediction_values.astype(np.float32),
    vertex_owner=np.asarray(mesh_prediction_projection["vertex_owner"], dtype=np.int64),
    mesh_path=str(mesh_prediction_projection["mesh_path"]),
    curvature_limit=np.float32(mesh_prediction_limit),
)


/tmp/ipykernel_656282/2441658616.py:17: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(str(FIGURES_DIR / f"{stem}.{suffix}"), scale=scale)


**Save Settings And Compact Results**

In [30]:
settings = {
    "created_at": RUN_TIMESTAMP,
    "project_root": PROJECT_ROOT,
    "save_dir": SAVE_DIR,
    "dataset": {
        "name": DATASET_NAME,
        "target_indices": TARGET_INDICES,
        "target_index_for_analysis": TARGET_INDEX_FOR_ANALYSIS,
        "marker_names": marker_names,
    },
    "exemplar": {
        "key": EXEMPLAR_KEY,
        "organoid_str": getattr(exemplar_graph, "organoid_str", None),
        "graph_path": exemplar_meta.get("graph_path"),
        "mesh_path": exemplar_meta.get("mesh_path"),
        "coordinate_source": coordinate_source,
        "preserve_forced_exemplar": PRESERVE_FORCED_EXEMPLAR,
    },
    "filtering": {
        "timepoint_filter_mode": TIMEPOINT_FILTER_MODE,
        "day3p5_timepoint": DAY3P5_TIMEPOINT,
        "filter_blacklisted_organoids": FILTER_BLACKLISTED_ORGANOIDS,
        "blacklist_n_keys": len(blacklist),
        "fill_missing_complexity": FILL_MISSING_COMPLEXITY,
        "missing_complexity_group": MISSING_COMPLEXITY_GROUP,
        "sphericity_max": SPHERICITY_MAX,
        "complexity_min": COMPLEXITY_MIN,
        "allow_missing_complexity": ALLOW_MISSING_COMPLEXITY,
        "interpolate_target_outliers": INTERPOLATE_TARGET_OUTLIERS,
        "outlier_clip_quantiles": OUTLIER_CLIP_QUANTILES,
        "n_graphs_after_filtering": len(graphs),
    },
    "split": {
        "val_frac": VAL_FRAC,
        "split_seed": SPLIT_SEED,
        "forced_val_keys": FORCED_VAL_KEYS,
        "forced_val_keys_found": split_info.get("forced_val_keys"),
        "train_keys": split_info.get("train_keys"),
        "val_keys": split_info.get("val_keys"),
    },
    "model": {
        "class": "GINCurvature",
        "depth": MODEL_DEPTH,
        "hidden_dim": HIDDEN_DIM,
        "dropout": DROPOUT,
        "norm": NORM,
        "residual": RESIDUAL,
        "use_global_features": USE_GLOBAL_FEATURES,
    },
    "training": {
        "lr": LR,
        "batch_size": BATCH_SIZE,
        "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE,
        "num_workers": NUM_WORKERS,
        "edge_loss_weight": EDGE_LOSS_WEIGHT,
        "edge_loss_params": EDGE_LOSS_PARAMS,
    },
    "graph_plot": {
        "node_size": GRAPH_NODE_SIZE,
        "edge_width": GRAPH_EDGE_WIDTH,
        "width": GRAPH_WIDTH,
        "height": GRAPH_HEIGHT,
        "camera": GRAPH_CAMERA,
        "normalize_coordinates": GRAPH_NORMALIZE_COORDINATES,
        "marker_threshold": MARKER_THRESHOLD,
        "encoding_color_mode": ENCODING_COLOR_MODE,
        "embedding_pca1_explained_variance": float(embedding_pca.explained_variance_ratio_[0]),
    },
    "mesh_plot": {
        "figure": "exemplar_mesh_by_predicted_curvature",
        "width": GRAPH_WIDTH,
        "height": GRAPH_HEIGHT,
        "camera": GRAPH_CAMERA,
        "colorscale": "RdBu_r",
        "center_curvature_at_zero": True,
        "curvature_limit": mesh_prediction_limit,
    },
}

for filename in ("settings.json", "config.json"):
    with open(SAVE_DIR / filename, "w") as handle:
        json.dump(jsonable(settings), handle, indent=2)

with open(SAVE_DIR / "results.pkl", "wb") as handle:
    pickle.dump({
        "split_summary": split_summary_df,
        "training_summary": training_summary_df,
        "marker_graph": marker_graph_df,
        "encoding_scores": encoding_df,
        "mesh_prediction_vertices": mesh_prediction_df,
        "settings": settings,
    }, handle)

print(f"Saved GNN illustration outputs to {SAVE_DIR}")


Saved GNN illustration outputs to /home/fmoller/Projects/LearningOrganoids/GraphNN/results_experiments/GNN_illustration/depth2_20250929_day4p5_B06_49_20260701_112827
